# Installment Count Analysis (ApplicationDate)

This notebook contains cleaned helpers and pipeline steps for loan counts by installment.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# Input datasets
data_2023 = pd.read_csv('/Users/starsrain/2025_concord/yieldCurve_augmenting/raw_data/2023_2025_data_0210.csv')
temp_df = pd.read_excel('/Users/starsrain/feb2026_concord/feb2026_harvey_files/Install_Paidoff_Raw.xlsx')

print('data_2023 shape:', data_2023.shape)
print('temp_df shape:', temp_df.shape)

In [ ]:
def prepare_df_strict_application(df):
    """Prepare dataframe with strict data cleaning using ApplicationDate."""
    out = df.copy()
    out = out[out['PaymentStatus'] != 'R']

    # Convert date columns
    for col in ['ApplicationDate', 'DueDate']:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors='coerce')

    # Drop missing critical fields
    out = out.dropna(subset=['LoanID', 'OriginatedAmount', 'InstallmentNumber'])

    # App month for cohorting
    out['AppMonth'] = out['ApplicationDate'].dt.to_period('M').astype(str)

    # Ensure CustType exists
    if 'CustType' not in out.columns:
        out['CustType'] = 'Unknown'

    # De-duplicate identical payment records
    out = out.drop_duplicates(subset=['LoanID', 'InstallmentNumber', 'DueDate', 'PaidOffPaymentAmount'])
    return out

In [ ]:
def create_cohort_loan_counts_by_installment_application(df, app_months=None):
    """Strict version: remove R, dedupe, filter DueDate <= today, then count unique loans."""
    prepared = prepare_df_strict_application(df)

    today = pd.Timestamp.now().normalize()
    filtered = prepared[prepared['DueDate'] <= today].copy()

    if app_months is not None:
        filtered = filtered[filtered['AppMonth'].isin(app_months)]

    filtered['Year'] = filtered['ApplicationDate'].dt.year
    filtered['Frequency'] = filtered['Frequency'].replace({'MB': 'M', 'ME': 'M'})

    counts = (
        filtered.groupby(['Year', 'CustType', 'Frequency', 'InstallmentNumber'], as_index=False)
        .agg(total_loans=('LoanID', 'nunique'))
        .rename(columns={'InstallmentNumber': 'Install_num'})
        .sort_values(['Year', 'CustType', 'Frequency', 'Install_num'])
        .reset_index(drop=True)
    )
    return counts

In [ ]:
def create_cohort_loan_counts_by_installment_application_raw_dynamic(df, cap_df, app_months=None):
    """
    Raw version: no strict cleaning. Cap InstallmentNumber dynamically by
    max Paidoff in cap_df for each (Year, CustType, Frequency).
    """
    raw = df.copy()
    raw['ApplicationDate'] = pd.to_datetime(raw['ApplicationDate'], errors='coerce')
    if 'DueDate' in raw.columns:
        raw['DueDate'] = pd.to_datetime(raw['DueDate'], errors='coerce')

    raw['AppMonth'] = raw['ApplicationDate'].dt.to_period('M').astype(str)
    if app_months is not None:
        raw = raw[raw['AppMonth'].isin(app_months)]

    raw['Year'] = raw['ApplicationDate'].dt.year
    raw['Frequency'] = raw['Frequency'].replace({'MB': 'M', 'ME': 'M'})

    cap_source = cap_df.copy()
    cap_source['Frequency'] = cap_source['Frequency'].replace({'MB': 'M', 'ME': 'M'})
    cap_max = (
        cap_source.groupby(['Year', 'CustType', 'Frequency'], as_index=False)['Paidoff']
        .max()
        .rename(columns={'Paidoff': 'MaxInstallment'})
    )

    raw = raw.merge(cap_max, on=['Year', 'CustType', 'Frequency'], how='left')
    raw['CapInstallment'] = raw['MaxInstallment'].fillna(raw['InstallmentNumber'])
    raw['Install_num'] = raw[['InstallmentNumber', 'CapInstallment']].min(axis=1)

    counts = (
        raw.groupby(['Year', 'CustType', 'Frequency', 'Install_num'], as_index=False)
        .agg(total_loans=('LoanID', 'nunique'))
        .sort_values(['Year', 'CustType', 'Frequency', 'Install_num'])
        .reset_index(drop=True)
    )
    return counts

In [ ]:
# Build app months from data_2023
raw_app_months_2023 = sorted(
    pd.to_datetime(data_2023['ApplicationDate'], errors='coerce')
    .dt.to_period('M')
    .astype(str)
    .dropna()
    .unique()
)

# Strict counts
app_cohort_counts_strict = create_cohort_loan_counts_by_installment_application(
    data_2023,
    app_months=raw_app_months_2023,
)
app_cohort_counts_strict.head()

In [ ]:
# Raw dynamic-cap counts (uses sample cap structure from temp_df)
app_cohort_counts_raw_dynamic = create_cohort_loan_counts_by_installment_application_raw_dynamic(
    data_2023,
    temp_df,
    app_months=raw_app_months_2023,
)
app_cohort_counts_raw_dynamic.head()